In [1]:
import os
import sys
from os import path

import time
import logging
import argparse
import re
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
import math
from statistics import mean
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import Draw
import zipfile
from jinja2 import Environment, PackageLoader
import numpy as np
import matplotlib.cm as cm
import glob



import timeit

import getWatersData
import cpAssignment
import genOutput
import plotting

In [2]:
def getUserReadableWell(wellno, plate_col_no):
    """
    Converts the well as a number into a user-friendly string,
    e.g. well 11 becomes "B5" for a 4*6 well plate. 

    :param wellno: An integer representing a specific well on the plate
    
    :return: A string representing a specific well on the plate
    """
    
    rowVal = math.floor((wellno-1) / plate_col_no)
    colVal = (wellno) % plate_col_no
    if colVal == 0:
        colVal = plate_col_no
    
    label = f'{chr(ord("@")+(rowVal)+1)}{colVal}'
    return label

def generateMol(smiles, name, save_dir):
    """
    Generates a 2D rendering of the given structure and saves 
    it as a .png

    :param smiles: a string (SMILES) of the compound
    :param name: a string corresponding to name of that compound
    :param save_dir: a string corresponding to the output directory
    
    :return: A .png rendering of the given compound
    """
    if smiles != "":
        mol = Chem.MolFromSmiles(smiles.strip())
        _discard = AllChem.Compute2DCoords(mol)
        Draw.MolToFile(mol, f'{save_dir}structures/{name}.png', size=(200, 150))

def buildHTML(save_dir, cpTable, analysis_name, times = {}):
    """
    Build a HTML output file using jinja2 and a html_template
    that is stored in the directory "templates". 
    
    :param save_dir: A string designating the output directory
    :param cpTable: Pandas datatable containing all information on 
                        the compounds used for analysis
    :param all_compounds: a list of all compound names
    :param impurities: a list of all impurity names
    :param analysis_name: User provided name for the analysis
    :param times: Optional parameter of a list of floats related to processing time
                    for each step of the analysis. 
    :return: HTML file saved to save_dir 
    """


    
    env = Environment(
        loader = PackageLoader("PyParse", "templates")
    )

    template = env.get_template("html_template.html")

    cptablerows = cpTable.to_dict(orient="records")
    

    with open(f'{save_dir}/html_output.html', "w") as fo:
        fo.write(template.render(
            cpnames = list(cpTable.loc[cpTable["type"] != "Impurity", "name"]),
            imp_no = len(cpTable.loc[cpTable["type"] == "Impurity"].index),
            cptablerows = cptablerows, 
            save_dir = save_dir,
            path = path,
            times = times,
            round = round,
            pt = "corrP/STD", 
            analysis_name = analysis_name,
            options = {}
            )
        )
    fo.close() 

In [3]:
#Create the save directory if one isn't already present.
save_dir = "testing/Performance Testing/"

error_msg = ""
try:
    os.mkdir(save_dir)
except OSError as error:
    error_msg = f'The directory "{save_dir}" already exists.'

    #make sub-directories to store all graphs and pictures of structures
try: 
    os.mkdir(f'{save_dir}/graphs')
except OSError as error:
    logging.debug("Graphs directory already exists.")
try: 
    os.mkdir(f'{save_dir}/structures')
except OSError as error:
    logging.debug("Structures directory already exists.")

In [4]:
timings = {}

In [5]:
rawData = getWatersData.rawWatersData("example_dataset/Waters/Example2/LC-MS Data for 48-Well Plate.rpt")

In [6]:
#Determine how the file records the well (e.g. "A,1", 1, "1,1", etc)
#Note that this must be done before processing the data
rawData.getWellFormat() 

In [7]:
#Process the data types in turn
rawData.processDAD()
rawData.processMS()
rawData.processUV()
rawData.processTrace(100)
rawData.processELSD()

In [8]:
#Select the lcData we want to use for this analysis and trim the traceData accordingly 
lcData = rawData.rawDADTable
traceData = rawData.rawTraceTable.loc[rawData.rawTraceTable["detector_type"] == "DAD"]

msData = rawData.rawMSTable
uvData = rawData.rawUVTable

In [9]:
cpTable = cpAssignment.Assignment("example_dataset/Waters/Example2/PyParse_designer_platemap.csv", 12, 8)
cpTable.generateCPTable()
cpTable.generateEMs("True") #Generate exact masses for each compound

cpTable.cpTable["type"] = pd.Categorical(cpTable.cpTable["type"], ["Product", "Reactant", "InternalSTD", "Byproduct"])
cpTable.cpTable.sort_values("type", inplace=True)

C:\Users\nosam\OneDrive\Documents\GitHub\PyParse\cpAssignment.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  self.inputCSV.fillna("", inplace=True)


In [10]:
#Find hits for each compound
cpTable.findHits(msData, lcData)

In [11]:
cpTable.validateHits(lcData, msData, uvData)

In [12]:
cpTable.removeDupAssigns(lcData, "mass_conf")

In [13]:
cpTable.findImpurities(lcData, msData, save_dir)

C:\Users\nosam\OneDrive\Documents\GitHub\PyParse\cpAssignment.py:212: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values("time", inplace = True)
C:\Users\nosam\OneDrive\Documents\GitHub\PyParse\cpAssignment.py:1039: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result.sort_values("time", inplace = True)
C:\Users\nosam\OneDrive\Documents\GitHub\PyParse\cpAssignment.py:1039: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result.

In [14]:
output = genOutput.Output(rawData.sample_IDs, 12, 8)
output.generateOutputTable(cpTable.cpTable, lcData)

In [15]:
cpTable.setBestWell(output.df, lcData, "corrP/STD")
cpTable.setBestMS(lcData)
cpTable.setBestTime(lcData)
cpTable.setBestPurity(lcData)

In [16]:
cpTable.findPotentialConflicts()

In [17]:
plotting.plotHeatmaps(output.df, save_dir, 12, 8)

In [18]:
byproducts = cpTable.cpTable.loc[cpTable.cpTable["type"] == "Byproduct", "name"]
plotting.plotPieCharts("corrP/STD", output.df, save_dir, byproducts, 12, 8)

C:\Users\nosam\OneDrive\Documents\GitHub\PyParse\plotting.py:672: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  labels.append(by_products[i])
C:\Users\nosam\OneDrive\Documents\GitHub\PyParse\plotting.py:672: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  labels.append(by_products[i])


In [19]:
timings["validationgraph"] = %timeit -o cpTable.cpTable.apply(plotting.plotHitValidationGraph, args = (lcData, save_dir, 12, 8,), axis = 1)

933 ms ± 16.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
timings["chroma"] = %timeit -o cpTable.cpTable.apply(plotting.plotChroma, args=(cpTable.cpTable, lcData, msData, traceData, save_dir,12), axis = 1)

2.48 s ± 16.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
plotting.plotHistogram(output.df, save_dir)
plotting.plotDonut(output.df, save_dir)

#Generate a set of PNG files to depict each compound
cpTable.cpTable.apply(lambda row: generateMol(row["canonSMILES"], row["name"], save_dir), axis = 1)

#Generate a location map for each compound
timings["locHeatmaps"] = %timeit -o cpTable.cpTable.apply(plotting.genLocationHeatmaps, args=(save_dir, 12, 8,), axis = 1)


63.2 ms ± 748 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
20.7 ms ± 101 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
8.6 ms ± 73.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
373 ms ± 9.68 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
timings["overlap"] = %timeit -o cpTable.findOverlap(lcData)

10.4 ms ± 218 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
#build the HTML output file
timings["buildHTML"] = %timeit -o buildHTML(save_dir, cpTable.cpTable, "v=3.7.15", times = {})

33.9 ms ± 294 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [24]:
timings

{'getWellformat': <TimeitResult : 480 μs ± 10.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)>,
 'processDAD': <TimeitResult : 43 ms ± 748 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
 'processMS': <TimeitResult : 89.8 ms ± 683 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
 'processUV': <TimeitResult : 22 ms ± 427 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
 'processTrace': <TimeitResult : 69.4 ms ± 1.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
 'processELSD': <TimeitResult : 20.5 ms ± 325 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
 'genCPTable': <TimeitResult : 27.1 ms ± 346 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
 'genEMs': <TimeitResult : 1.56 ms ± 31.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)>,
 'findHits': <TimeitResult : 720 ms ± 8.08 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)>,
 'validateHits': <TimeitResult : 115 ms ± 2.39 ms per loop (mean ± std. de

In [25]:
timings_list = {}
for index, timing in timings.items():
    print(timing)
    timings_list[index] = mean(timing.timings)
    

480 μs ± 10.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
43 ms ± 748 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
89.8 ms ± 683 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
22 ms ± 427 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
69.4 ms ± 1.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
20.5 ms ± 325 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
27.1 ms ± 346 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.56 ms ± 31.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
720 ms ± 8.08 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
115 ms ± 2.39 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.55 ms ± 14.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
6.14 ms ± 299 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
104 ms ± 706 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
6.08 ms ± 51.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
928 μs ± 18.9 μs

In [26]:
timings_list

{'getWellformat': 0.0004801444428571488,
 'processDAD': 0.043029964285714026,
 'processMS': 0.08981946999999764,
 'processUV': 0.021960911428576375,
 'processTrace': 0.06939049571428119,
 'processELSD': 0.02054865857142626,
 'genCPTable': 0.0271335385714305,
 'genEMs': 0.0015647508714285194,
 'findHits': 0.7198604000000485,
 'validateHits': 0.11475302857142457,
 'remDupAssigns': 0.0015495053428571087,
 'findImps': 0.0061386142856788084,
 'genOutput': 0.1037426228571446,
 'setBestWell': 0.006079133428571432,
 'setBestMS': 0.000928123685714289,
 'setBestTime': 0.004187263714284849,
 'setBestPurity': 0.00416931985714265,
 'conflicts': 0.008933083142857347,
 'heatmap': 0.9033760142857058,
 'piecharts': 1.2843106714285568,
 'validationgraph': 0.8969771428571676,
 'chroma': 2.38438778571432,
 'histogram': 0.06324899571428172,
 'donut': 0.020746761428573075,
 'drawMols': 0.008597980142857134,
 'locHeatmaps': 0.3731740571428678,
 'overlap': 0.010378003000000392,
 'buildHTML': 0.033902565714281

In [27]:
timing_df = pd.DataFrame.from_dict(timings_list, orient="index")

Please refer to https://modin.readthedocs.io/en/stable/supported_apis/defaulting_to_pandas.html for explanation.
INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:51331
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:51340'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:51348'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:51356'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:51334'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:51342'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:51346'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:5135

In [28]:
timing_df

,0
getWellformat,0.000480
processDAD,0.043030
processMS,0.089819
processUV,0.021961
processTrace,0.069390
processELSD,0.020549
genCPTable,0.027134
genEMs,0.001565
findHits,0.719860
validateHits,0.114753


In [29]:
timing_df.to_csv("testing/Performance Testing/multicore_modin_3.13.7.csv")

In [24]:
len(traceData.loc[traceData["well"] == 5].index)

87